# Checkpoint 39: Feature Diagnostics

This notebook reviews redundancy, reference-category encoding, VIF, condition number, and employee-cluster bootstrap coefficient stability. Target-informed results use only the 2023 and 2024 development snapshots.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

In [ ]:
inventory = pd.read_csv(PROCESSED_DIR / "feature_diagnostics.csv")
numeric = pd.read_csv(PROCESSED_DIR / "numeric_correlation_diagnostics.csv")
categorical = pd.read_csv(PROCESSED_DIR / "categorical_association_diagnostics.csv")
vif = pd.read_csv(PROCESSED_DIR / "vif_diagnostics.csv")
condition = pd.read_csv(PROCESSED_DIR / "condition_number_diagnostics.csv")
manifest = pd.read_csv(PROCESSED_DIR / "encoded_feature_manifest.csv")
coefficients = pd.read_csv(PROCESSED_DIR / "coefficient_stability.csv")
checks = pd.read_csv(PROCESSED_DIR / "feature_policy_validation.csv")
bootstrap = pd.read_csv(PROCESSED_DIR / "bootstrap_run_summary.csv")

## Validation checks

Every check should show `PASS`. The reserved 2025 target usage should equal zero.

In [ ]:
checks

## Complete feature policy

In [ ]:
inventory.groupby(["decision", "role"], as_index=False).size()

In [ ]:
inventory[["feature", "decision", "role", "reason"]]

## Strongest original numeric correlations

These candidate-feature relationships explain why several raw columns were removed.

In [ ]:
(
    numeric.loc[numeric["stage"].eq("candidate")]
    .sort_values("absolute_correlation", ascending=False)
    .head(15)
)

## Strongest original categorical associations

In [ ]:
categorical.head(15)

## Variance inflation factors

The policy requires every encoded VIF to remain at or below 5.

In [ ]:
top_vif = vif.head(15).sort_values("vif")

plt.figure(figsize=(9, 6))
plt.barh(top_vif["encoded_feature"], top_vif["vif"], color="#4C78A8")
plt.axvline(5.0, color="#E45756", linestyle="--", label="Configured limit")
plt.xlabel("Variance inflation factor")
plt.title("Highest selected-feature VIF values")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
condition

## Reference-category manifest

Each categorical coefficient compares one displayed category with its omitted reference category.

In [ ]:
manifest

## Employee-cluster bootstrap coefficient stability

Only directions whose 95% bootstrap interval excludes zero and whose sign stability is at least 90% are displayed. These are predictive associations, not causal effects.

In [ ]:
stable = coefficients.loc[
    coefficients["reportable_direction"].astype(bool)
].copy()
stable[
    [
        "encoded_feature",
        "bootstrap_median",
        "bootstrap_lower_95",
        "bootstrap_upper_95",
        "sign_stability",
        "direction",
    ]
]

In [ ]:
plot_data = stable.sort_values("bootstrap_median")
lower_error = plot_data["bootstrap_median"] - plot_data["bootstrap_lower_95"]
upper_error = plot_data["bootstrap_upper_95"] - plot_data["bootstrap_median"]

plt.figure(figsize=(10, 6))
plt.errorbar(
    plot_data["bootstrap_median"],
    plot_data["encoded_feature"],
    xerr=[lower_error, upper_error],
    fmt="o",
    color="#4C78A8",
    ecolor="#72B7B2",
    capsize=3,
)
plt.axvline(0, color="black", linestyle="--", linewidth=1)
plt.xlabel("Regularized logistic coefficient")
plt.title("Reportable employee-cluster bootstrap directions")
plt.tight_layout()
plt.show()

## Bootstrap scope

In [ ]:
bootstrap

## Conclusion

The selected reference-encoded matrix is full rank, has acceptable VIF and condition number, and preserves the 2025 holdout. Coefficient stability is treated as predictive evidence only. Checkpoint 40 will define the train, validation, and test periods.